In [11]:
import numpy as np
from tqdm import tqdm
import scipy.stats as sp
from scipy.special import gamma
from scipy.optimize import minimize
from scipy.integrate import nquad 
from scipy.sparse import block_diag, diags
from jupyprint import jupyprint, arraytex
from numdifftools import Hessian, Jacobian
from IPython.display import clear_output
import matplotlib.pyplot as plt

In [12]:
# Ginzburg Landau model check and test 
#parameters: 

p = 5
d = p**3
tau=2.0
lamb=0.5
alpha=0.1

def potential(X):
    X = np.reshape(X, (p,p,p))
    temp = np.linalg.norm(np.roll(X, -1, axis=0) - X)**2 \
            + np.linalg.norm(np.roll(X, -1, axis=1) - X)**2 \
            + np.linalg.norm(np.roll(X, -1, axis=2) - X)**2
    return 0.5*(1-tau)*np.linalg.norm(X)**2 + 0.5*tau*alpha*temp \
            + (1./4)*tau*lamb*np.sum(np.power(X,4))


def gradient(X):
    X = np.reshape(X, (p,p,p))
    temp = np.roll(X, 1, axis=0)+np.roll(X, -1, axis=0) \
            +np.roll(X, 1, axis=1)+np.roll(X, -1, axis=1) \
            +np.roll(X, 1, axis=2)+np.roll(X, -1, axis=2)
    gradU = (1.0-tau)*X + tau*lamb*np.power(X,3) + tau*alpha*(6*X - temp)
    return gradU.flatten()

small_matrix = np.diag(np.ones(p-1),1) +  np.diag(np.ones(p-1),-1)
small_matrix[0,p-1] += 1
small_matrix[p-1,0] += 1

middle_diagonal = block_diag([small_matrix]*p)
middle_diagonal += diags(np.ones(p*(p-1)), p)
middle_diagonal += diags(np.ones(p), (p-1)*p)
middle_diagonal += diags(np.ones(p*(p-1)), -p)
middle_diagonal += diags(np.ones(p), -(p-1)*p)

MIXED_DER_MATR = block_diag([middle_diagonal]*p)
MIXED_DER_MATR += diags(np.ones(p**2 *(p-1)), p**2)
MIXED_DER_MATR += diags(np.ones(p**2 *(p-1)), -p**2)
MIXED_DER_MATR += diags(np.ones(p**2), p**2 * (p-1))
MIXED_DER_MATR += diags(np.ones(p**2), -p**2 * (p-1))

def hessian(X): 
    output = np.eye(len(X)) * (1-tau + 6*tau*alpha) + np.diag(3*tau*lamb*X**2) - tau*alpha * MIXED_DER_MATR.toarray()
    return output

def hessian_p(X, p): 
    output = p * (1-tau + 6*tau*alpha) + p * 3*tau*lamb*X**2 - tau*alpha * MIXED_DER_MATR * p  + 1/time_step * p
    return output

a=0

In [13]:
# PARAMETERS OF THE RUN
initial_sample = np.zeros(d)
time_step = 5e-5
sampling = 10**4
n_MC = 100
tune_interval = 10000
number_of_samples = sampling+tune_interval

In [14]:
# EXPLICIT TAMED SCHEME

# Monte Carlo 
moment2_explicit_tamed = []
moment4_explicit_tamed = []
moment6_explicit_tamed = []

for i_MC in range(0, n_MC):
    print("MC: "+str(i_MC+1)+"/"+str(n_MC))

    samples_explicit_tamed = [initial_sample]
    samples_explicit_tamed_norm = [np.linalg.norm(initial_sample)]

    for _ in tqdm(range(1, number_of_samples)):

        # gradient tamed step
        x = samples_explicit_tamed[-1] - time_step * gradient(samples_explicit_tamed[-1]) / (1+time_step*np.linalg.norm(gradient(samples_explicit_tamed[-1]))) 
        
        # adding Gaussian
        cov = np.diag(np.repeat(2*time_step,d))
        x = sp.multivariate_normal.rvs(mean=x, cov=cov)
        # save value
        samples_explicit_tamed.append(x)
        samples_explicit_tamed_norm.append(np.linalg.norm(x))

    samples_explicit_tamed_norm = np.array(samples_explicit_tamed_norm[tune_interval:])
    samples_explicit_tamed = np.array(samples_explicit_tamed[tune_interval:])

    moment2_explicit_tamed.append(np.mean(samples_explicit_tamed_norm**2))
    moment4_explicit_tamed.append(np.mean(samples_explicit_tamed_norm**4))
    moment6_explicit_tamed.append(np.mean(samples_explicit_tamed_norm**6))

jupyprint("explicit tamed")
jupyprint("2nd: mean = " + str(np.mean(moment2_explicit_tamed)) + ", std = " + str(np.std(moment2_explicit_tamed)))
jupyprint("4th: mean = " + str(np.mean(moment4_explicit_tamed)) + ", std = " + str(np.std(moment4_explicit_tamed)))
jupyprint("6th: mean = " + str(np.mean(moment6_explicit_tamed)) + ", std = " + str(np.std(moment6_explicit_tamed)))

MC: 1/100


100%|██████████| 19999/19999 [01:18<00:00, 253.37it/s]


MC: 2/100


100%|██████████| 19999/19999 [01:31<00:00, 218.73it/s]


MC: 3/100


...

100%|██████████| 19999/19999 [01:07<00:00, 296.92it/s]


explicit tamed

2nd: mean = 76.38414810432033, std = 5.640973348393622

4th: mean = 5902.228656694588, std = 869.4239614033282

6th: mean = 461210.01666931267, std = 101827.2027940889

In [15]:
# EXPLICIT SCHEME 

# Monte Carlo 
moment2_explicit = []
moment4_explicit = []
moment6_explicit = []

for i_MC in range(0, n_MC):
    print("MC: "+str(i_MC+1)+"/"+str(n_MC))

    samples_explicit = [initial_sample]
    samples_explicit_norm = [np.linalg.norm(initial_sample)]

    for _ in tqdm(range(1, number_of_samples)):

        #gradient step
        x = samples_explicit[-1] - time_step * gradient(samples_explicit[-1])  
        
        # adding Gaussian
        cov = np.diag(np.repeat(2*time_step,d))
        x = sp.multivariate_normal.rvs(mean=x, cov=cov)
        # save value
        samples_explicit.append(x)
        samples_explicit_norm.append(np.linalg.norm(x))

    samples_explicit_norm = np.array(samples_explicit_norm[tune_interval:])
    samples_explicit = np.array(samples_explicit[tune_interval:])

    moment2_explicit.append(np.mean(samples_explicit_norm**2))
    moment4_explicit.append(np.mean(samples_explicit_norm**4))
    moment6_explicit.append(np.mean(samples_explicit_norm**6))

jupyprint("explicit")
jupyprint("2nd: mean = " + str(np.mean(moment2_explicit)) + ", std = " + str(np.std(moment2_explicit)))
jupyprint("4th: mean = " + str(np.mean(moment4_explicit)) + ", std = " + str(np.std(moment4_explicit)))
jupyprint("6th: mean = " + str(np.mean(moment6_explicit)) + ", std = " + str(np.std(moment6_explicit)))

MC: 1/100


100%|██████████| 19999/19999 [00:58<00:00, 341.32it/s]


MC: 2/100


100%|██████████| 19999/19999 [01:02<00:00, 320.80it/s]


MC: 3/100


...

100%|██████████| 19999/19999 [00:59<00:00, 337.97it/s]


explicit

2nd: mean = 77.4903219076883, std = 5.400196258328786

4th: mean = 6079.065452783285, std = 850.2952812568993

6th: mean = 482759.731873103, std = 102211.82122426902

In [16]:
# IMPLICIT INEXACT SCHEME 

# Monte Carlo 
moment2_implicit = []
moment4_implicit = []
moment6_implicit = []

for i_MC in range(0, n_MC):
    clear_output(wait=True)
    print("MC: "+str(i_MC+1)+"/"+str(n_MC))

    samples_implicit = [initial_sample]
    samples_implicit_norm = [np.linalg.norm(initial_sample)]

    for _ in tqdm(range(1, number_of_samples)):

        # inexact proximal step 
        x = minimize(
            lambda x: potential(x) + 1/(2*time_step) * np.linalg.norm(x - samples_implicit[-1])**2, 
            jac=lambda x: gradient(x) + 1/time_step * (x - samples_implicit[-1]),
            hessp=hessian_p,
            method="Newton-CG",
            x0=samples_implicit[-1]
            ).x
        
        # adding Gaussian
        cov = np.diag(np.repeat(2*time_step,d))
        x = sp.multivariate_normal.rvs(mean=x, cov=cov)
        # save value
        samples_implicit.append(x)
        samples_implicit_norm.append(np.linalg.norm(x))

    samples_implicit = np.array(samples_implicit[tune_interval:])
    samples_implicit_norm = np.array(samples_implicit_norm[tune_interval:])

    moment2_implicit.append(np.mean(samples_implicit_norm**2))
    moment4_implicit.append(np.mean(samples_implicit_norm**4))
    moment6_implicit.append(np.mean(samples_implicit_norm**6))

jupyprint("implicit")
jupyprint("2nd: mean = " + str(np.mean(moment2_implicit)) + ", std = " + str(np.std(moment2_implicit)))
jupyprint("4th: mean = " + str(np.mean(moment4_implicit)) + ", std = " + str(np.std(moment4_implicit)))
jupyprint("6th: mean = " + str(np.mean(moment6_implicit)) + ", std = " + str(np.std(moment6_implicit)))

MC: 100/100


100%|██████████| 19999/19999 [01:13<00:00, 270.75it/s]


implicit

2nd: mean = 77.31897742111016, std = 6.078973675562712

4th: mean = 6050.73529762711, std = 942.9689822619922

6th: mean = 479093.13697471813, std = 111271.76358582563